In [9]:
from tdc.single_pred import Tox
data = Tox(name = 'ClinTox')
split = data.get_split(method='scaffold')


Found local copy...
Loading...
Done!
100%|██████████| 1478/1478 [00:00<00:00, 3133.81it/s]


In [10]:
print(split.keys())
split["train"].columns


dict_keys(['train', 'valid', 'test'])


Index(['Drug_ID', 'Drug', 'Y'], dtype='object')

In [11]:
split["train"].head()

,Drug_ID,Drug,Y
0,Drug 15,CC(C)C[C@H](NC(=O)CNC(=O)c1cc(Cl)ccc1Cl)B(O)O,1
1,Drug 1231,CCCCCCC[NH+](CC)CCCC(O)c1ccc(NS(C)(=O)=O)cc1,0
2,Drug 1052,CC[C@@H](c1cccc(O)c1)[C@@H](C)C[NH+](C)C,0
3,Drug 1073,CC[N+](C)(C)c1cccc(O)c1,0
4,Drug 1074,CC[N+](C)(C)Cc1ccccc1Br,0


In [12]:
for k in split:
    print(f"{k}:")
    print(split[k]["Y"].value_counts())
    print("total:", len(split[k]))
    print()


train:
Y
0    963
1     71
Name: count, dtype: int64
total: 1034

valid:
Y
0    130
1     17
Name: count, dtype: int64
total: 147

test:
Y
0    273
1     24
Name: count, dtype: int64
total: 297



In [13]:
data = Tox(name="ClinTox")
df = data.get_data()
len(df)
data = Tox(name="ClinTox")
df = data.get_data()
len(df)


Found local copy...
Loading...
Done!
Found local copy...
Loading...
Done!


1478

In [14]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.metrics import average_precision_score, roc_auc_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

df_clintox = pd.read_csv("data/clintox_rdkit.csv")

train_df = df_clintox[df_clintox["split"] == "train"]
valid_df = df_clintox[df_clintox["split"] == "valid"]
test_df  = df_clintox[df_clintox["split"] == "test"]

feat_cols = [c for c in df_clintox.columns if c not in ["Y", "split", "Drug", "Drug_ID"]]

X_train = train_df[feat_cols].replace([np.inf, -np.inf], np.nan).to_numpy()
y_train = train_df["Y"].to_numpy()
X_valid = valid_df[feat_cols].replace([np.inf, -np.inf], np.nan).to_numpy()
y_valid = valid_df["Y"].to_numpy()
X_test  = test_df[feat_cols].replace([np.inf, -np.inf], np.nan).to_numpy()
y_test  = test_df["Y"].to_numpy()

# νέο combined και split για το search
X_combined = np.vstack([X_train, X_valid])
y_combined = np.concatenate([y_train, y_valid])
split_index = [-1] * len(X_train) + [0] * len(X_valid)
ps = PredefinedSplit(split_index)

print(f"Train: {(y_train==1).sum()} θετικά / {(y_train==0).sum()} αρνητικά")
print(f"Valid: {(y_valid==1).sum()} θετικά / {(y_valid==0).sum()} αρνητικά")
print(f"Test:  {(y_test==1).sum()} θετικά / {(y_test==0).sum()} αρνητικά")

Train: 71 θετικά / 963 αρνητικά
Valid: 17 θετικά / 130 αρνητικά
Test:  24 θετικά / 273 αρνητικά
